In [1]:
from data_engineering.flights.flight_type import get_flights, get_flights_from_iceberg
from datetime import date,timedelta
import polars as pl

start_date = date(2025,7,1)
end_date = date(2025,8,1)
df_flights = pl.DataFrame()
for target_date in [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]:
    df_flights = pl.concat([df_flights, get_flights(target_date)], how="vertical")
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""010058""","""SUBNP""","""SU-BNP""",2025-07-01 15:14:19.210,"""HEAL""",2025-07-01 15:31:18.240,"""HEOC""",false,false,false,false,"""GLF4""",null,"""GULFSTREAM 4""","""A2"""
"""0100a2""","""MSR551""","""SU-GCE""",2025-07-01 13:09:30.800,"""HE30""",2025-07-01 13:28:17.190,"""HECA""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR542""","""SU-GCE""",2025-07-01 18:23:43.610,"""HE42""",2025-07-01 20:25:19.410,"""LTFM""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR543""","""SU-GCE""",2025-07-01 21:48:55.030,"""LTFM""",2025-07-01 23:21:37.740,"""HEAZ""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a3""","""MSR552""","""SU-GCF""",2025-07-01 07:38:19.150,"""HECA""",2025-07-01 07:49:18.290,"""HE30""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e94c8a""","""BOV795""",null,2025-07-31 18:42:42.730,"""AR-0583""",2025-07-31 19:13:21.180,"""PY-0033""",null,null,null,null,null,null,null,"""A2"""
"""e94c8e""","""BOV767""","""CP-3214""",2025-07-31 02:25:36.910,"""KMIA""",2025-07-31 03:57:41.260,"""JM-0016""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""e94c8f""","""BOV736""","""CP-3215""",2025-07-31 10:44:41.510,"""SLLJ""",2025-07-31 15:39:38.710,"""SBGR""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""


In [2]:
df_flights = df_flights.filter((pl.col("ladd") == True) | (pl.col("pia") == True))

In [3]:
df_flights = df_flights.with_columns(
    pl.concat_str(
        [
            pl.col("icao"),
            pl.col("takeoff_time").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ],
        separator="_",
    ).alias("flight_id")
)

In [4]:
df_flights = df_flights.select("flight_id", "icao", "takeoff_time", "landing_time", "landing_airport_ident", "takeoff_airport_ident", "ladd", "pia", "category")

In [5]:
next_ids = (
    df_flights.join(
        df_flights.select(
            pl.col("flight_id").alias("next_flight_id"),
            pl.col("icao").alias("next_icao"),
            pl.col("pia").alias("next_pia"),
            pl.col("category").alias("next_category"),
            "takeoff_airport_ident",
            pl.col("takeoff_time").alias("next_takeoff_time"),
        ),
        left_on="landing_airport_ident",
        right_on="takeoff_airport_ident",
        how="left",
    )
    .filter(
        (pl.col("next_takeoff_time") > pl.col("landing_time"))
        & (
            (pl.col("next_icao") == pl.col("icao"))
            | (
                pl.col("next_pia")
                & pl.col("category").is_not_null()
                & pl.col("next_category").is_not_null()
                & (pl.col("next_category") == pl.col("category"))
            )
        )
    )
    .group_by("flight_id")
    .agg(
        pl.col("next_flight_id")
        .unique()
        .alias("next_flight_ids")
    )
)

df = (
    df_flights
    .join(next_ids, on="flight_id", how="left")
    .with_columns(
        pl.col("next_flight_ids")
        .fill_null(pl.lit([], dtype=pl.List(pl.String)))
    )
)

In [6]:
import networkx as nx

G = nx.DiGraph()

# Add all flights as nodes
for row in df.iter_rows(named=True):
    G.add_node(
        row["flight_id"],
        icao=row["icao"],
        takeoff_time=row["takeoff_time"],
        landing_time=row["landing_time"],
        takeoff_airport_ident=row["takeoff_airport_ident"],
        landing_airport_ident=row["landing_airport_ident"],
        ladd=row["ladd"],
        pia=row["pia"],
    )

# Add edges to possible next flights
for row in df.select(["flight_id", "next_flight_ids"]).iter_rows(named=True):
    src = row["flight_id"]

    for dst in row["next_flight_ids"]:
        G.add_edge(src, dst)

In [7]:
import networkx as nx

# Keep only connected regions that include both PIA and LADD flights
pia_nodes = {
    row["flight_id"]
    for row in df.filter(pl.col("pia")).select("flight_id").iter_rows(named=True)
}

ladd_nodes = {
    row["flight_id"]
    for row in df.filter(pl.col("ladd")).select("flight_id").iter_rows(named=True)
}

# Find every node connected to a PIA node
keep_nodes = set()

for pia_node in pia_nodes:
    keep_nodes.update(nx.descendants(G, pia_node))
    keep_nodes.update(nx.ancestors(G, pia_node))
    keep_nodes.add(pia_node)

G = G.subgraph(keep_nodes).copy()

keep_components = []
for component in nx.weakly_connected_components(G):
    if component & pia_nodes and component & ladd_nodes:
        keep_components.extend(component)

# Subgraph containing only regions that can produce mixed PIA/LADD paths
G = G.subgraph(keep_components).copy()

In [8]:
G.number_of_nodes()

60988

In [9]:
subgraphs = [
    G.subgraph(c).copy()
    for c in nx.weakly_connected_components(G)
]
len(subgraphs)

29

In [10]:
def path_has_pia_and_ladd(path):
    return (
        any(G.nodes[node].get("pia") for node in path)
        and any(G.nodes[node].get("ladd") for node in path)
    )

def _same_flag_icao(node, icao, flag):
    node_data = G.nodes[node]
    return node_data.get(flag) and node_data.get("icao") == icao

def _uses_longest_continuous_runs(path, flag):
    i = 0

    while i < len(path):
        node_data = G.nodes[path[i]]

        if not node_data.get(flag):
            i += 1
            continue

        icao = node_data.get("icao")
        j = i + 1

        while j < len(path) and _same_flag_icao(path[j], icao, flag):
            j += 1

        first_node = path[i]
        last_node = path[j - 1]
        previous_node = path[i - 1] if i > 0 else None
        next_node = path[j] if j < len(path) else None

        can_extend_before = any(
            _same_flag_icao(predecessor, icao, flag)
            and predecessor != previous_node
            for predecessor in G.predecessors(first_node)
        )
        can_extend_after = any(
            _same_flag_icao(successor, icao, flag)
            and successor != next_node
            for successor in G.successors(last_node)
        )

        if can_extend_before or can_extend_after:
            return False

        i = j

    return True

def uses_longest_continuous_pia_and_ladd_runs(path):
    return (
        _uses_longest_continuous_runs(path, "pia")
        and _uses_longest_continuous_runs(path, "ladd")
    )

paths = []
seen = set()
MAX_PATHS = 500

for component in nx.weakly_connected_components(G):
    component_graph = G.subgraph(component).copy()
    source_nodes = [
        node
        for node in component_graph.nodes
        if component_graph.in_degree(node) == 0
    ]

    if not source_nodes:
        source_nodes = list(component_graph.nodes)

    for src in source_nodes:
        shortest_paths = nx.single_source_shortest_path(component_graph, src)

        for dst, path in shortest_paths.items():
            if src == dst:
                continue

            key = tuple(path)

            if (
                key not in seen
                and path_has_pia_and_ladd(path)
                and uses_longest_continuous_pia_and_ladd_runs(path)
            ):
                paths.append(path)
                seen.add(key)

                if MAX_PATHS is not None and len(paths) >= MAX_PATHS:
                    break

        if MAX_PATHS is not None and len(paths) >= MAX_PATHS:
            break

    if MAX_PATHS is not None and len(paths) >= MAX_PATHS:
        break

paths

[['a8ff1b_2025-07-14 18:52:21',
  'a8ff1b_2025-07-15 14:32:31',
  'a8ff1b_2025-07-15 18:09:01',
  'a8ff1b_2025-07-15 19:33:54',
  'a03bbc_2025-07-31 11:25:22'],
 ['a8ff1b_2025-07-14 18:52:21',
  'a8ff1b_2025-07-15 14:32:31',
  'a8ff1b_2025-07-15 18:09:01',
  'a8ff1b_2025-07-15 19:33:54',
  'a03ccf_2025-07-29 11:36:19',
  'a03ccf_2025-07-29 18:45:28'],
 ['a24373_2025-07-01 11:04:51', 'a03b4c_2025-07-28 14:17:56'],
 ['a3a868_2025-07-01 23:12:10',
  'a3a868_2025-07-02 19:40:29',
  'a3a868_2025-07-02 22:55:21',
  'a100dc_2025-07-07 14:26:09'],
 ['a1f526_2025-07-08 21:31:15',
  'a1f526_2025-07-11 11:57:33',
  'a03b4c_2025-07-28 14:17:56'],
 ['a63559_2025-07-14 14:01:44',
  'a63559_2025-07-15 11:37:01',
  'a03bbc_2025-07-31 11:25:22'],
 ['a63559_2025-07-14 14:01:44',
  'a63559_2025-07-15 11:37:01',
  'a03ccf_2025-07-29 11:36:19',
  'a03ccf_2025-07-29 18:45:28'],
 ['aae2ca_2025-07-07 00:08:20',
  'aae2ca_2025-07-07 01:37:08',
  'aae2ca_2025-07-08 13:52:51',
  'a03b4c_2025-07-28 14:17:56'],
 [

In [13]:
# Test Graph

In [11]:
from data_engineering.flights.flight_type import Flight, flights_df_to_flights
flights_objs:dict[str, Flight] = dict()
for flight in df_flights.iter_rows(named=True):
     flights_objs[flight["flight_id"]] = Flight(**flight)

In [12]:
df_path_0 = pl.DataFrame()
for flight_id in paths[3]:
    flight = flights_objs[flight_id]
    df_path_0 = pl.concat(
        [
            df_path_0,
            pl.DataFrame(
                {
                    "flight_id": flight.flight_id,
                    "icao": flight.icao,
                    "takeoff_time": flight.takeoff_time,
                    "landing_time": flight.landing_time,
                    "takeoff_airport_ident": flight.takeoff_airport_ident,
                    "landing_airport_ident": flight.landing_airport_ident,
                    "ladd": flight.ladd,
                    "pia": flight.pia,
                    "category": flight.category,
                }
            ),
        ],
        how="vertical",
    )
df_path_0

flight_id,icao,takeoff_time,landing_time,takeoff_airport_ident,landing_airport_ident,ladd,pia,category
str,str,datetime[μs],datetime[μs],str,str,bool,bool,str
"""a3a868_2025-07-01 23:12:10""","""a3a868""",2025-07-01 23:12:10.250,2025-07-01 23:49:32.570,"""KSTL""","""KIND""",true,false,"""A2"""
"""a3a868_2025-07-02 19:40:29""","""a3a868""",2025-07-02 19:40:29.470,2025-07-02 20:09:51.940,"""KIND""","""KLEX""",true,false,"""A2"""
"""a3a868_2025-07-02 22:55:21""","""a3a868""",2025-07-02 22:55:21.770,2025-07-02 23:11:21.220,"""KLEX""","""KSDF""",true,false,"""A2"""
"""a100dc_2025-07-07 14:26:09""","""a100dc""",2025-07-07 14:26:09.650,2025-07-07 15:40:22.710,"""KSDF""","""08GA""",false,true,"""A2"""


In [ ]:
paths[0]

['a10151_2025-07-26 19:31:31', 'a10151_2025-07-31 16:21:28']

In [ ]:
a = set(df_flights.filter(pl.col("pia") == True).get_column("icao").unique())
"a03bc7" in a

True

In [ ]:
# I should make dict with key being flight_id and value being Flight Object 

In [ ]:
# how do I deal with getting back to the icaos. 
# I can iterate for through each flight
# I'm going to try with assuuming flight data is perfect first just to see how far I get. 